# Welcome To The Pluto Realty Database!

    * User Guide Written by: 
        Anthony Sesay
        5/12/16

## Description:

This is a database application for Pluto Realty, Inc. (Pluto) to help it in managing the rental of real estate properties across the country.  

The intended users of the application are the employees of Pluto. It is assumed that for this guide this notebook will be ran in VS code or in browser vis JupyterLab and access to MySQL server.

Now let's get started

## Step 1: Setup

First This code segemnt will conect your client to your MySQL server to get started.

All you need to do is to run this code block and enter your user and password for your mysql server.

In [ ]:
%pip install jupysql sqlalchemy pymysql mysql-connector-python

%load_ext sql
import getpass

db_user = getpass.getpass('Enter username: ')
db_password = getpass.getpass('Enter password: ')
db_host = 'localhost'

# Create DB using a temporary connector 
import mysql.connector

con = mysql.connector.connect(
    host=db_host,
    user=db_user,
    password=db_password
)

cursor = con.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS PlutoRealty")
cursor.close()
con.close()

# Connect SQL magic to the database
import os
os.environ["DATABASE_URL"] = f"mysql+pymysql://{db_user}:{db_password}@{db_host}/PlutoRealty"
%sql

## Step 2: Creating and Loading The Database

Now that you are able to connect this notebook to a database, run the three SQL scripts in MySQL Workbench or anything that can interact with databases

- **`1. createAll.sql`** — Creates all tables, constraints, and triggers
- **`2. loadAll.sql`** — Inserts all sample data into the tables

The dropALL file isn't needed to create the database, but it is useful for deleting tables and reseting

- **`dropAll.sql`** — Drops any existing tables so the setup starts fresh

Similarly, queryAll isn't required either, but used for running specifc statements in your workbench

- **'queryAll'**
- 

After that step, now the Database is ready to be used. Below are some example activities to show how to use it and a report for users

### Activities 

To write a query begin the code block with %%sql and then write the statement you wish to execute. The following are example activites you can do



In [ ]:
%%sql
SHOW TABLES;

1. Edit the details/characteristics of a given property.  


In [ ]:
%%sql 
UPDATE RentalProperty SET rent = 1500.00 WHERE property_number = 1;

2. Create a new client. 

In [ ]:
%%sql
INSERT INTO Person (id, p_name, address, phone, email) VALUES
(30, 'Fred Flintstone', '123 Main St, Anytown, USA', '123-456-7890', 'fredflintstone@email.com');
INSERT INTO Clients (id) VALUES (30);




3. Enter information for a new property viewing. 


In [ ]:
%%sql
INSERT INTO ViewProperty (client_id, associate_id, property_number, view_time) VALUES
(30, 4, 10, '2026-05-15 14:00:00');

4. Discard a given property (and all its associated information). 

- To delete a tuple, first you want to delete from the relation tables that reference it, then entities that reference it and lastly itself

In [ ]:
%%sql
DELETE FROM Contract WHERE property_number = 10;
DELETE FROM ViewProperty WHERE property_number = 10;
DELETE FROM Owns WHERE property_number = 10;
DELETE FROM Lease WHERE property_number = 10;
DELETE FROM RentalProperty WHERE property_number = 10;

5. Change the partner assigned to an owner and transfer all current (non-expired) 
leases of that owner to the new partner. 
    o Use a SQL transaction to ensure ACID properties (serializability) in the 
    face of other concurrent operations or failures.  

In [ ]:
%%sql
-- view all contracts for properties owned by owner_id 19 
SELECT 
    partner_id,
    Lease.lease_number,
    Owns.owner_id

FROM Contract
JOIN Owns ON Contract.property_number = Owns.property_number
JOIN Lease ON Contract.lease_number = Lease.lease_number
           AND Contract.property_number = Lease.property_number
WHERE Owns.owner_id = 19
ORDER BY Contract.lease_number;

In [ ]:
%%sql
START TRANSACTION;

-- Scripting variables
SET @owner_id = 19;
SET @old_partner_id = 1;
SET @new_partner_id = 2;

-- Insert new contract rows with new partner for all active leases
INSERT INTO Contract (partner_id, lease_number, client_id, property_number)
SELECT 
    @new_partner_id,
    Lease.lease_number,
    Contract.client_id,
    Contract.property_number
FROM Contract
JOIN Owns ON Contract.property_number = Owns.property_number
JOIN Lease ON Contract.lease_number = Lease.lease_number
           AND Contract.property_number = Lease.property_number
WHERE Owns.owner_id      = @owner_id
    AND Contract.partner_id    = @old_partner_id
    AND Lease.finish        >= CURDATE();

-- Delete old contract rows
DELETE Contract FROM Contract
JOIN Owns ON Contract.property_number = Owns.property_number
JOIN Lease ON Contract.lease_number = Lease.lease_number
           AND Contract.property_number = Lease.property_number
WHERE Owns.owner_id      = @owner_id
  AND Contract.partner_id    = @old_partner_id
  AND Lease.finish        >= CURDATE();

COMMIT;

In [ ]:
%%sql

-- verify the update by viewing all contracts for properties owned by owner_id 19, 
SELECT 
    partner_id,
    Lease.lease_number,
    Owns.owner_id

FROM Contract
JOIN Owns ON Contract.property_number = Owns.property_number
JOIN Lease ON Contract.lease_number = Lease.lease_number
            AND Contract.property_number = Lease.property_number
WHERE Owns.owner_id = 19
ORDER BY Contract.lease_number;

### Reports 
1. List the names of all (unique) clients. 


In [ ]:
%%sql

SELECT DISTINCT p_name from Clients join Person on Clients.id = Person.id; 


2. Find the unique names of owners and total square footage of all the properties 
they own. 


In [ ]:
%%sql
SELECT DISTINCT p_name, sum(area) 
from Owns 
	JOIN Person on Owns.owner_id = Person.id
	JOIN RentalProperty on Owns.property_number = RentalProperty.property_number
GROUP BY Person.p_name;


3. Find the properties shown by each associate in a given month.  


In [ ]:
%%sql
SELECT 
    MONTH(view_time) AS view_month,
    property_number,
    COUNT(*) AS num_viewings
FROM ViewProperty 
WHERE view_time >= '2026-05-01' 
    AND view_time < '2026-06-02'
GROUP BY MONTH(view_time), property_number
ORDER BY num_viewings DESC;

4. Find the most popular properties (in terms of number of viewings) in a given 
period (range of dates).  



In [ ]:
%%sql
SELECT 
    ViewProperty.property_number,
    address,
    type,
    COUNT(*) AS viewings
FROM ViewProperty
JOIN RentalProperty ON ViewProperty.property_number = RentalProperty.property_number
WHERE view_time >= '2026-01-01' 
    AND view_time <  '2026-07-02'
GROUP BY property_number, address, type
ORDER BY viewings DESC;

5. Find the total rent due to each property owner in a given month and year. 


In [ ]:
%%sql
SELECT 
    owner_id,
    p_name,
    SUM(Lease.rent) AS total_rent
FROM Lease
JOIN Owns ON Lease.property_number = Owns.property_number
JOIN Person ON Owns.owner_id = Person.id
WHERE YEAR(Lease.start)  = 2024
    AND MONTH(Lease.start) = 5
GROUP BY Owns.owner_id, Person.p_name;

6. Find the unique names of clients that were ever shown at least three or more 
unique residential properties owned by a given owner.  




In [ ]:
%%sql
SELECT 
    client_id,
    p_name
FROM ViewProperty
JOIN Owns  ON ViewProperty.property_number = Owns.property_number
JOIN RentalProperty ON ViewProperty.property_number = RentalProperty.property_number
JOIN Person ON ViewProperty.client_id = Person.id
WHERE Owns.owner_id = 19
  AND RentalProperty.type = 'Residential'
GROUP BY ViewProperty.client_id, Person.p_name
HAVING COUNT(DISTINCT ViewProperty.property_number) >= 3;

7. Find the unique names of owners that have a residential property in every city 
where Pat Doe owns a commercial property. 



In [ ]:
%%sql
SELECT 
    owner_id,
    p_name
FROM Owns
JOIN RentalProperty ON Owns.property_number = RentalProperty.property_number
JOIN Person ON Owns.owner_id = Person.id
WHERE RentalProperty.type = 'residential'
-- get the city from the address and check if it's in the list of cities where Pat Doe has commercial properties
    AND SUBSTRING_INDEX(RentalProperty.address, ',', -1) IN (
    SELECT DISTINCT SUBSTRING_INDEX(RentalProperty.address, ',', -1)
    FROM RentalProperty
    JOIN Owns ON RentalProperty.property_number = Owns.property_number
    JOIN Person ON Owns.owner_id = Person.id
    WHERE Person.p_name = 'Pat Doe'
    AND RentalProperty.type = 'commercial'
)
-- group by owner and name, and only include those who have properties in all the cities where Pat Doe has commercial properties
GROUP BY Owns.owner_id, Person.p_name
HAVING COUNT(DISTINCT SUBSTRING_INDEX(RentalProperty.address, ',', -1)) = (
    SELECT COUNT(DISTINCT SUBSTRING_INDEX(RentalProperty.address, ',', -1))
    FROM RentalProperty
    JOIN Owns ON RentalProperty.property_number = Owns.property_number
    JOIN Person ON Owns.owner_id = Person.id
    WHERE Person.p_name = 'Pat Doe'
    AND RentalProperty.type = 'commercial'
);

8. Find the top-3 partners with respect to number of properties leased in the current 
year. 



In [ ]:
%%sql
SELECT 
    partner_id,
    p_name,
    COUNT(DISTINCT Lease.property_number) AS properties_leased
FROM Contract
JOIN Lease  ON Contract.lease_number = Lease.lease_number 
            AND Contract.property_number = Lease.property_number
JOIN Person ON Contract.partner_id = Person.id
WHERE YEAR(Lease.start) = 2026
GROUP BY Contract.partner_id, Person.p_name
ORDER BY properties_leased DESC
LIMIT 3;

9. total management fees due to Pluto in the last 3 months.  


In [ ]:
%%sql
SELECT 
    SUM(management_fee) AS total_management_fees
FROM RentalProperty 
JOIN Lease ON RentalProperty.property_number = Lease.property_number
WHERE start >= DATE_SUB(CURDATE(), INTERVAL 3 MONTH)
    AND start <  CURDATE();

### Congrats!! You've reached the end

Now that you've completed the tutorial, you're free to start quering to your heart's content. Stay tuned for the SQL

In [ ]:
#statements go here
%%sql



